# 🎮 SheepBell - Google Colab Startup Notebook
Automatic Video Clipping & Transcription Tool for Game LQA (Gradio UI)

### 📌 Workflow Overview
1. **Runtime Setup**: Ensure that **T4 GPU** is selected under **"Runtime" > "Change runtime type"** in the menu.
2. **Run cells 1 to 3 sequentially**: Open the generated `https://xxxx.gradio.live` link to process the video.
3. **Run cell 4 (Post-processing)**: Automatically creates a dated folder in Google Drive, saves the artifacts (video clips, images, CSV), and generates a Spreadsheet in one click.

### 1. (Recommended) Mount Google Drive
Run this first to allow access, so the generated artifacts (clips, images, spreadsheet) can be saved to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Clone / Update Repository (Clone for the first time, pull for subsequent times)

In [ ]:
import os

# SheepBell GitHub repository
REPO_URL = "https://github.com/sheep-works/SheepBell"
TARGET_DIR = "/content/SheepBell"

if not os.path.exists(TARGET_DIR):
    print("🚀 Cloning repository...")
    !git clone {REPO_URL} {TARGET_DIR}
else:
    print("🔄 Updating existing repository (git pull)...")
    %cd {TARGET_DIR}
    !git pull

%cd {TARGET_DIR}

# Install dependencies (including GPU libraries)
!pip install -q gradio faster-whisper silero-vad onnxruntime-gpu imageio-ffmpeg gspread nvidia-cublas-cu12 nvidia-cudnn-cu12

### 3. Launch Web UI (Generate Gradio Share URL)
When executed, `Running on public URL: https://xxxx.gradio.live` will be displayed at the bottom. Click the link to use the tool in your browser.
*(※ Keep this cell running while using the UI)*

In [ ]:
# Launch Gradio UI on GPU and issue a public URL
!python app.py --share

---
## 🎁 Post-processing Tool After Completion
Run the cell below after the processing in the Web UI is finished.

### 4. 🚀 Save all artifacts (Videos, Images, CSV) to Google Drive & Auto-create a Spreadsheet
Copies all artifacts from Colab's temporary storage to the specified Google Drive folder, while automatically creating a Google Spreadsheet and issuing a link.

In [ ]:
import os
import csv
import shutil
import datetime
from pathlib import Path
from google.colab import auth
from google.auth import default
import gspread
from googleapiclient.discovery import build

# ==========================================
# Settings: Destination folder name in Google Drive
# ==========================================
DRIVE_BASE_DIR = "/content/drive/MyDrive/LQA_Results"
SRC_ISSUES_DIR = "/content/SheepBell/issues"

# 1. Authenticate Google Account
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive_service = build('drive', 'v3', credentials=creds)

# Helper function to get or create folder in Google Drive
def get_or_create_drive_folder(service, folder_name, parent_id=None):
    query = f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    if parent_id:
        query += f" and '{parent_id}' in parents"
    results = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
    files = results.get('files', [])
    if files:
        return files[0]['id']
    file_metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder'}
    if parent_id:
        file_metadata['parents'] = [parent_id]
    folder = service.files().create(body=file_metadata, fields='id').execute()
    return folder.get('id')

# 2. Create base folder and dated subfolder in Google Drive
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
base_folder_id = get_or_create_drive_folder(drive_service, "LQA_Results")
subfolder_name = f"LQA_{timestamp}"
target_folder_id = get_or_create_drive_folder(drive_service, subfolder_name, parent_id=base_folder_id)

target_drive_folder = Path(DRIVE_BASE_DIR) / subfolder_name
target_drive_folder.mkdir(parents=True, exist_ok=True)

# 3. Copy all generated files (Video clips, Images, CSV, JSON) to Google Drive
if os.path.exists(SRC_ISSUES_DIR):
    for item in os.listdir(SRC_ISSUES_DIR):
        s = os.path.join(SRC_ISSUES_DIR, item)
        d = os.path.join(str(target_drive_folder), item)
        if os.path.isfile(s):
            shutil.copy2(s, d)
    print(f"📁 All files saved to Google Drive: {target_drive_folder}")
else:
    print("⚠️ 'issues' folder not found.")

# 4. Create Google Spreadsheet directly inside the target folder
csv_path = target_drive_folder / "lqa_issues.csv"
if not csv_path.exists():
    csv_path = Path(SRC_ISSUES_DIR) / "lqa_issues.csv"

if csv_path.exists():
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))

    sheet_title = f"LQA_Issues_{timestamp}"
    sh = gc.create(sheet_title, folder_id=target_folder_id)
    worksheet = sh.get_worksheet(0)
    worksheet.update(rows)

    print("\n🎉 Google Spreadsheet created in the target folder!")
    print(f"🔗 Open Spreadsheet -> {sh.url}")
else:
    print("⚠️ CSV file not found.")

### 5. 📦 (Optional) Download all as Zip to local PC
Run this if you want to download a Zip directly to your local PC without using Google Drive.

In [ ]:
!zip -r /content/issues.zip /content/SheepBell/issues
from google.colab import files
files.download("/content/issues.zip")